# Compile cNMF Evaluation Results into Excel

This notebook collects all evaluation outputs (enrichment, perturbation association, explained variance, etc.) for a given cNMF run and compiles them into a single multi-sheet Excel workbook per (K, threshold) combination.

**Prerequisites**: You should have already run the cNMF evaluation pipeline (see `cNMF_evaluation_pipeline.ipynb`).

## Pipeline Steps

| Step | Description |
|------|-------------|
| **Step 1. Set Parameters** | Define I/O paths, MuData keys, K/threshold grid, and sample labels |
| **Step 2. Load Guide Data** | Load guide assignment AnnData and define helper to attach guide metadata |
| **Step 3. Compile Excel** | For each (K, threshold): load MuData, gather all evaluation DataFrames, build summary sheets, and write to `.xlsx` |

## Expected Output

One Excel file per (K, threshold): `{out_dir}/{run_name}/Eval/{K}_{thresh}/cNMF_{K}_{thresh}.xlsx`

### Output Sheets

| Sheet | Rows | Description | Key Columns |
|-------|------|-------------|-------------|
| **Summary** | 1 per program | Overview of each program with top genes, enrichment highlights, perturbation hit counts, and mean scores per sample | `program_name`, `top10_loaded_genes`, `top30_loaded_genes`, `Automatic Timepoint`, `Total Enriched GO Terms`, `Significant regulators with positive/negative effect {sample}`, `sigfdr0.05_targets_sorted_abslog2fcd_{sample}`, `Top 5 specific regulators (FDR<0.1) {sample}`, `Mean program score {sample}`, `top10_enriched_genesets`, `top10_enriched_go_terms`, `manual_annotation_label` (user fills in) |
| **Program Loadings** | top N genes x programs | Long-format gene loading scores with gene descriptions | `Program`, `Rank`, `Gene`, `Annotation` (NCBI/RefSeq description) |
| **Targets Summary** | 1 per target | Per-target aggregated perturbation stats: expression, cell counts, significant programs, correlated targets | `target_name`, `mean_expression_{sample}`, `# Cells {sample}`, `significant programs {sample}`, `# programs {sample}`, `top 5 specific programs (FDR < 0.1) {sample}`, `top 5 pos/neg correls targets (program log2fc) {sample}` |
| **Sample Association** | 1 per program | Kruskal-Wallis test + Dunn's posthoc for program score differences across samples | `program_name`, `batch_kruskall_wallis_pval`, `{sample}_batch_association_dunn_min_pval`, `{sample}_batch_association_dunn_mean_pval` |
| **Perturbation Association** | 1 per target x program x sample | Full Mann-Whitney U test results (split across multiple sheets if >1M rows) | `target_name`, `program_name`, `specificity_scores`, `beta_hat`, `p-value`, `adj_pval` (Storey Q), `approx_log2FC`, `Sample` |
| **Significant Regulators Only** | subset of above | Same as Perturbation Association, filtered to `adj_pval < 0.05` | Same columns as Perturbation Association |
| **Trait Enrichment** | 1 per term x program | GWAS trait enrichment via Fisher exact test against Open Targets L2G | `Term`, `program_name`, `P-value`, `Adjusted P-value`, `Odds Ratio`, `Combined Score`, `Genes`, `overlap_numerator`, `overlap_denominator` |
| **GO Term Enrichment** | 1 per term x program | GO Biological Process 2023 enrichment via Fisher exact test | Same columns as Trait Enrichment |
| **Geneset Enrichment** | 1 per term x program | Reactome 2022 pathway enrichment via Fisher exact test | Same columns as Trait Enrichment |

In [1]:
import pandas as pd
import muon as mu
import sys
import os

sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from Stage3_Interpretation.B_Summarization.src import (
    compile_Program_loading_score_sheet_long, compile_Program_loading_score_sheet_flat,
    Compile_GO_sheet, Compile_Geneset_sheet, Compile_Trait_sheet,
    Compile_Perturbation_sheet, Compile_Association_sheet, Compile_Explained_variance,
    Compile_Target_Summary_sheet, Compile_Summary_sheet,
    add_specificity_scores_file, check_program_name_match
)

/home/users/ymo/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1. Set Parameters — Define I/O paths, MuData keys, K/threshold grid, and sample labels


In [8]:

# ── cNMF output paths ──
out_dir = '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result'
run_name = '030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50'


# ── MuData keys ──
prog_key = 'cNMF'                          # modality key for cNMF programs
data_key = 'rna'                           # modality key for RNA expression
gene_names_key = 'symbol'                  # var column for gene symbols (used for loading scores and target summary)
categorical_key = 'batch'              # obs column for sample/condition labels
guide_targets_key = "guide_targets"        # uns key for guide target names

# ── Compilation settings ──
num_gene = 300                             # number of top genes per program to include in loadings
components = [50]                          # K values to compile
sel_threshs = [0.2]                        # density thresholds to compile
batch_values = ['D0','D1','D2','D3']       # sample/condition labels (must match categorical_key values)
perturbation_file_name = 'CRT'             # prefix for perturbation result files (e.g. 'CRT' or 'perturbation_association_results')
non_targeting_key = ['non-targeting']       # guide target labels used as negative controls
effect_size = 'approx_log2FC'                     # column name for effect size in perturbation results

# ── Column name keys (match your enrichment / perturbation TSV headers) ──
GO_Term_key = "Term"                       # index column in GO enrichment file
GO_Genes_key = "Genes"                     # gene list column in GO enrichment file
Geneset_Term_key = "Term"                  # index column in geneset enrichment file
Geneset_Genes_key = "Genes"                # gene list column in geneset enrichment file
Trait_Term_key = "Term"                    # index column in trait enrichment file
Trait_Genes_key = "Genes"                  # gene list column in trait enrichment file
Perturbation_Sample_key = "Sample"         # column name for sample labels in perturbation results
adjusted_pval_key = "Adjusted P-value"     # column name for adjusted p-value in enrichment files

## Step 2. Compile Excel — For each (K, threshold), gather all evaluation results and write to .xlsx

Sub-steps per iteration:
* 3a. Load MuData and attach guide metadata
* 3b. Load individual evaluation result DataFrames (loadings, GO, Reactome, trait, perturbation, etc.)
* 3c. Validate that program names are consistent across all DataFrames
* 3d. Build Target Summary (aggregated perturbation effects per target across samples)
* 3e. Build Summary sheet (one row per program with highlights from all evaluations)
* 3f. Write all sheets to a single Excel workbook (see Output Sheets table above for details)


In [ ]:
for sel_thresh in sel_threshs:
    for k in components:

        thresh_str = str(sel_thresh).replace('.', '_')
        eval_base = f'{out_dir}/{run_name}/Eval/{k}_{thresh_str}'
        output_folder = f'{out_dir}/{run_name}/Interpretation/Summary_table/{k}_{thresh_str}'
        os.makedirs(output_folder, exist_ok=True)


        # ── 2a. Load MuData (guide data already embedded) ──
        mdata_path = f'{out_dir}/{run_name}/adata/cNMF_{k}_{thresh_str}.h5mu'
        print(f'Loading MuData: {mdata_path}')
        mdata = mu.read(mdata_path)

        # ── 2b. Load evaluation DataFrames from Eval/ ──
        GO_path = f'{eval_base}/{k}_GO_term_enrichment.txt'
        Geneset_path = f'{eval_base}/{k}_geneset_enrichment.txt'
        Trait_path = f'{eval_base}/{k}_trait_enrichment.txt'
        Perturbation_path_base = f'{eval_base}/{k}_{perturbation_file_name}'
        Association_path = f'{eval_base}/{k}_categorical_association_results.txt'
        Explained_Variance_path = f'{eval_base}/{k}_Explained_Variance.txt'

        # Program loadings
        df_Program_loading_long = compile_Program_loading_score_sheet_long(mdata, num_gene=num_gene, data_key=data_key, gene_names_key=gene_names_key)
        df_Program_loading_flat = compile_Program_loading_score_sheet_flat(mdata, num_gene=num_gene, data_key=data_key, gene_names_key=gene_names_key)

        # Enrichment sheets
        df_GO = Compile_GO_sheet(GO_path, term_key=GO_Term_key, genes_key=GO_Genes_key) if os.path.exists(GO_path) else None
        df_Geneset = Compile_Geneset_sheet(Geneset_path, term_key=Geneset_Term_key, genes_key=Geneset_Genes_key) if os.path.exists(Geneset_path) else None
        df_Trait = Compile_Trait_sheet(Trait_path, term_key=Trait_Term_key, genes_key=Trait_Genes_key) if os.path.exists(Trait_path) else None

        # Perturbation (WTC only)
        perturbation_files = [f'{Perturbation_path_base}_{samp}.txt' for samp in batch_values]
        if any(os.path.exists(f) for f in perturbation_files):
            df_Perturbation = Compile_Perturbation_sheet(Perturbation_path_base, Sample=batch_values, sample_key=Perturbation_Sample_key)
            df_Perturbation_significant_gene_only = df_Perturbation[df_Perturbation['adj_pval'] < 0.05]
        else:
            print(f'No perturbation files found for: {Perturbation_path_base}')
            df_Perturbation = None
            df_Perturbation_significant_gene_only = None

        # Categorical association & explained variance
        df_Association = Compile_Association_sheet(Association_path) if os.path.exists(Association_path) else None
        df_Explained_Variance = Compile_Explained_variance(Explained_Variance_path) if os.path.exists(Explained_Variance_path) else None

        # ── 2c. Validate program names ──
        check_program_name_match(mdata, prog_key=prog_key, dataframes=[
            df_GO, df_Geneset, df_Trait, df_Perturbation,
            df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only
        ])

        # ── 2d. Build Target Summary ──
        # Uses WTC perturbation + batch-level cell counts/expression
        df_Target_Summary = Compile_Target_Summary_sheet(
            mdata, Perturbation_path_base,
            Sample=batch_values, categorical_key=categorical_key,
            prog_key=prog_key, data_key=data_key,
            guide_targets_key=guide_targets_key,
            save_path=output_folder, effect_size=effect_size,
            gene_names_key=gene_names_key
        )

        # ── 2e. Build Summary sheet ──
        # Pass batch_values for 'Automatic Timepoint' (mean scores are per-batch);
        # perturbation columns use df_Perturbation['Sample'].unique() = ['WTC'] internally
        df_Summary = Compile_Summary_sheet(
            mdata, df_GO, df_Geneset, df_Perturbation, df_Program_loading_flat, df_Explained_Variance,
            Sample=batch_values, specicicity_path=output_folder,
            categorical_key=categorical_key, non_tagerting_key=non_targeting_key,
            effect_size=effect_size, adjusted_pval_key=adjusted_pval_key
        )

        # ── 2e2. Save key DataFrames as separate TSV files ──
        df_Summary.to_csv(f'{output_folder}/Summary_{k}_{thresh_str}.tsv', sep='\t')
        df_Program_loading_long.to_csv(f'{output_folder}/Program_Loadings_{k}_{thresh_str}.tsv', sep='\t')
        df_Target_Summary.to_csv(f'{output_folder}/Targets_Summary_{k}_{thresh_str}.tsv', sep='\t')
        print(f'Saved separate TSV files to {output_folder}')

        # ── 2f. Write Excel ──
        print(f'Compiling Excel for K={k}, threshold={sel_thresh}')
        MAX_ROWS = 1048575
        excel_path = f'{output_folder}/cNMF_{k}_{thresh_str}.xlsx'

        with pd.ExcelWriter(excel_path) as writer:
            df_Summary.to_excel(writer, sheet_name='Summary', index=True)
            df_Program_loading_long.to_excel(writer, sheet_name='Program Loadings', index=True)
            df_Target_Summary.to_excel(writer, sheet_name='Targets Summary', index=True)

            if df_Association is not None:
                df_Association.to_excel(writer, sheet_name='Sample Association', index=True)

            if df_Perturbation is not None:
                # Re-load with specificity scores
                combined_conditions = []
                for samp in batch_values:
                    df_Perturbation_ = add_specificity_scores_file(output_folder, Perturbation_path_base, samp)
                    df_Perturbation_[Perturbation_Sample_key] = samp
                    combined_conditions.append(df_Perturbation_)
                df_Perturbation_with_spec = pd.concat(combined_conditions)

                for i in range(0, len(df_Perturbation_with_spec), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_with_spec.iloc[i:i+MAX_ROWS].to_excel(
                        writer, sheet_name=f'Perturbation Association {sheet_num}', index=True)

                for i in range(0, len(df_Perturbation_significant_gene_only), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_significant_gene_only.iloc[i:i+MAX_ROWS].to_excel(
                        writer, sheet_name=f'significant regulators only {sheet_num}', index=True)

            if df_Trait is not None:
                df_Trait.to_excel(writer, sheet_name='Trait Enrichment', index=True)
            if df_GO is not None:
                df_GO.to_excel(writer, sheet_name='GO Term Enrichment', index=True)
            if df_Geneset is not None:
                df_Geneset.to_excel(writer, sheet_name='Geneset Enrichment', index=True)

        print(f'Done. Saved to {excel_path}')

Loading MuData: /oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu
Load program loadings data in long form


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

Load program loadings data in flat form
Load GO data
Load geneset data
Load trait data
Load perturbation data
Load categorical associa data
Generate target summary sheet
Get targeted gene mean expressuion per day
Compute guides cells per days
Compile significant programs
Compute significant programs
Calculate specificity scores for program
Compute correlation for D0
Compute correlation for D1
Compute correlation for D2
Compute correlation for D3
Compute KD efficiency per target per condition
